# 03 — Kiểm tra mô hình trước huấn luyện

    Chạy độc lập trên máy local hoặc Kaggle. Các bài thử dùng trọng số ngẫu nhiên và dữ liệu tổng hợp, **không huấn luyện mô hình chính**. Trên Kaggle đặt `REPO_REF` bằng SHA đã review. Cần Python 3.11, PyTorch, pytest, TensorBoard; local: `uv sync --group model --group train --group dev --group notebook`. Kaggle: kiểm tra PyTorch/CUDA cài sẵn rồi cài thiếu `pytest tensorboard ipykernel`, không thay torch CUDA bằng wheel CPU.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import torch

REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = ''  # Kaggle: đặt commit SHA đã review; không dùng main di động.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p/'src'/'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE:
    if len(REPO_REF) < 40:
        raise ValueError('Đặt REPO_REF bằng commit SHA 40 ký tự đã review trước khi chạy Kaggle')
    if not (REPO_DIR/'src'/'llm_reasoning').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
print('commit:', CODE_COMMIT, 'dirty:', DIRTY, 'torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
sys.path.insert(0, str(REPO_DIR/'src'))
os.environ['PYTHONPATH'] = str(REPO_DIR/'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
import importlib.util
for package in ('pytest', 'tensorboard'):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError(f'Cần cài {package} rồi chạy lại kernel')
if ON_KAGGLE and DIRTY:
    raise RuntimeError('Repo Kaggle phải sạch và đúng commit SHA')
from llm_reasoning.config import make_config
from llm_reasoning.model import TransformerLM
from llm_reasoning.tokenizer import ByteBPETokenizer
cfg = make_config('target')
print('Target config:', cfg.model, cfg.train)
print('Target parameters at vocab 8000:', TransformerLM(cfg.model, 8000).num_parameters())

## Kiểm thử cơ chế

    Forward/backward, causal mask, tokenizer, cache, training và checkpoint/resume. Kết quả phải xanh trước notebook 04. CPU FP16/CUDA tests được skip trên máy không có CUDA.

In [ ]:
tests = ['tests/test_layers.py', 'tests/test_attention.py', 'tests/test_model.py',
         'tests/test_generation.py', 'tests/test_kv_cache.py', 'tests/test_training.py',
         'tests/test_checkpoint.py', 'tests/test_training_data.py']
completed = subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], cwd=REPO_DIR, check=True)
print('Test return code:', completed.returncode)

## Overfit và resume trên dữ liệu tổng hợp

    Mỗi script xuất summary, metrics CSV và TensorBoard trong thư mục mới. Script resume so sánh run liền và run ngắt trên cùng CPU. Đây là hai bài kiểm tra độc lập, không phải hai mô hình ứng viên.

In [ ]:
from datetime import datetime, timezone
OUT = (Path('/kaggle/working') if ON_KAGGLE else REPO_DIR/'artifacts') / ('nb03_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
subprocess.run([sys.executable, str(REPO_DIR/'scripts/task_2_6_smoke.py'),
                '--output', str(OUT/'overfit')], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, str(REPO_DIR/'scripts/task_2_7_smoke.py'),
                '--output', str(OUT/'resume')], cwd=REPO_DIR, check=True)
summary = json.loads((OUT/'resume'/'summary.json').read_text(encoding='utf-8'))
print('Output:', OUT, '\nResume:', json.dumps(summary, indent=2))